In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import copy
import sys
import random
import torchvision
import torchvision.transforms as transforms
import torch
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.trainer import IntervalTrainer, InterContiNetTrainer
# Define transform (e.g., convert to tensor)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

from src.data_utils import split_mnist_by_labels, get_context_sets
from src.utils import general as utils
import src.models as models

import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

from src.utils.plotting_style import set_figure_size
from src.regulariser import L2Regulariser, UnbiasRegulariser, MultiRegulariser

In [3]:
device = "cuda"
classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

task_pairs = [("cat", "truck"), ("frog", "ship"), ("horse", "automobile"), ("dog", "airplane"), ("bird", "deer")]
task_pairs_ids = [(classes.index(i), classes.index(j)) for i, j in task_pairs]
animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1 ,0, 0]).to(device)

def domain_map_fn(y):
    return animals_mask[y]


@torch.no_grad()
def encode(x, model, device="cuda"):
    # Handle batching to avoid out-of-memory issues
    batch_size = 2048
    features = []
    for i in range(0, len(x), batch_size):
        batch = x[i : i + batch_size].to(device)
        batch_features = model(batch).flatten(start_dim=1).cpu()
        features.append(batch_features)
    return torch.cat(features, dim=0)


def encode_dataset(train_dataset, test_dataset, encoder, device="cuda"):
    train_imgs, train_labels = train_dataset.data, train_dataset.targets
    test_imgs, test_labels = test_dataset.data, test_dataset.targets
    # apply the appropriate scaling and transposition
    train_imgs = torch.tensor(train_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    test_imgs = torch.tensor(test_imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    train_imgs = (train_imgs - 0.5) / 0.5
    test_imgs = (test_imgs - 0.5) / 0.5
    train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
    test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()

    if encoder is not None:
        train_imgs = encode(train_imgs, encoder, device)
        test_imgs = encode(test_imgs, encoder, device)

    train_dataset = torch.utils.data.TensorDataset(train_imgs, train_labels)
    test_dataset = torch.utils.data.TensorDataset(test_imgs, test_labels)
    return train_dataset, test_dataset


def get_tasks(encoder):
    
    non_animals = [0, 1, 8, 9]
    animals = [2, 3, 4, 5, 6, 7]

    non_animal_indices = torch.tensor(non_animals)[torch.randperm(4)].tensor_split(5)
    animal_indices = list(torch.tensor(animals)[torch.randperm(6)].tensor_split(5))
    animal_indices.reverse()

    task_pairs_ids = [t.tolist() + n.tolist() for t, n in zip(animal_indices, non_animal_indices)]
    print("Contexts:",  task_pairs_ids)    
    trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
    trainset.targets = torch.tensor(trainset.targets)
    testset.targets = torch.tensor(testset.targets)

    if encoder is not None:
        trainset, testset = encode_dataset(trainset, testset, encoder)
    test_tasks = [split_mnist_by_labels(testset, task_pair_id) for task_pair_id in task_pairs_ids]
    train_tasks = [split_mnist_by_labels(trainset, task_pair_id) for task_pair_id in task_pairs_ids]

    return train_tasks, test_tasks

In [4]:
# Get the CIFAR 100 dataset
cifar100_trainset = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=transform)
cifar100_testset = torchvision.datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

# Convert targets to tensor for consistency
cifar100_trainset.targets = torch.tensor(cifar100_trainset.targets)
cifar100_testset.targets = torch.tensor(cifar100_testset.targets)

# Print dataset info
print(f"CIFAR-100 training set: {len(cifar100_trainset)} images")
print(f"CIFAR-100 test set: {len(cifar100_testset)} images")
print(f"Number of classes: {len(set(cifar100_trainset.targets.tolist()))}")

# Sample a few class names
print(f"Sample classes: {cifar100_trainset.classes[:10]}")

CIFAR-100 training set: 50000 images
CIFAR-100 test set: 10000 images
Number of classes: 100
Sample classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [5]:
# encoder = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT).to(device)
# encoder.fc = torch.nn.Linear(512, 100)
# encoder.to(device)
# # task0_train, task0_test = get_tasks(None)[0]
# pretrain_data_train, pretrain_data_test = cifar100_trainset, cifar100_testset
# simple_trainer = SimpleTrainer(encoder)
# simple_trainer.train(pretrain_data_train, pretrain_data_test, n_epochs=4, learning_rate=0.001, val_freq=200)
# base_model = encoder
# # Create a directory for saved models if it doesn't exist
# save_dir = os.path.join(project_root, "notebooks/saved_models")
# os.makedirs(save_dir, exist_ok=True)

# # Save the complete model
# model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")
# torch.save(base_model.state_dict(), model_path)

In [6]:
# Create a simple function to load the model
save_dir = os.path.join(project_root, "notebooks/saved_models")
model_path = os.path.join(save_dir, "resnet18_cifar100_pretrained.pth")
def load_pretrained_model(model_path, num_classes=100):
    model = torchvision.models.resnet18(weights=None)
    model.fc = torch.nn.Linear(512, num_classes)
    model.load_state_dict(torch.load(model_path))
    return model

model = load_pretrained_model(model_path)

In [7]:
encoder = copy.deepcopy(model).cuda()
encoder.fc = torch.nn.Identity()

In [8]:
X_min, X_max = None, None
for task in get_tasks(encoder):
    X, _ = next(iter(torch.utils.data.DataLoader(task[0], batch_size=10000, shuffle=False)))
    if X_min is None:
        X_min, X_max = X.min(dim=0).values, X.max(dim=0).values
    else:
        X_min = torch.min(X_min, X.min(dim=0).values)
        X_max = torch.max(X_max, X.max(dim=0).values)
X_min, X_max = X_min.to(device), X_max.to(device)

Contexts: [[6, 9], [2, 0], [4, 1], [7, 8], [5, 3]]


/tmp/ipykernel_2232659/2243627393.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_2232659/2243627393.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


In [9]:
from collections import defaultdict
from src.helpers.WandbWrapper import WandbTrainerWrapper
from src.models import get_mnist_model
from src.data_utils import get_mnist_tasks
from src.utils.general import set_seed

import wandb

results = {
    "class": defaultdict(list),
    "task": defaultdict(list),
    "domain": defaultdict(list),
}

In [14]:
def run_cil():
    config = {
        "ours": True,
        "init.projection_strategy": "best_loss",
        "init.n_certificate_samples": 400,
        "init.min_acc_limit": 1,
        "init.min_acc_increment": 0.2,
        "init.paradigm": "CIL",
        "init.n_iters": 200,
        "init.primal_learning_rate": 0.33,
        "init.dual_learning_rate": 0.01,
        "init.penalty_coefficient": 1,
        "init.checkpoint": 2,
        "train.l2_lambda": 0.01,
        "train.unbias_lambda": 0.01,
        "train.lr": 0.02,
        "train.weight_decay": 0,
        "train.epochs": 5,
        "train.batch_size": 128,
        "simple_train.epochs": 5,
        "simple_train.batch_size": 128,
        "simple_train.lr": 0.02,
        "simple_train.weight_decay": 0,
        "benchmarks": {
            "ewc": {
                "lmbd": 1e6,
                "fisher_batch": 64,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "sgd": {
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "lwf": {
                "lmbd": 0.1,
                "temp": 2,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "icn": {"lr": 0.01, "batch_size": 128, "epochs": 30, "lid_lr": 100},
        }
    }
    tag = "final_cifar_cil_new"
    benchmark_tags = [f"final_cifar_cil_{bench}" for bench in config["benchmarks"].keys()]

    for i in range(6, 10):
        set_seed(i)
        config["init.seed"] = i
        train_tasks, test_tasks = get_tasks(encoder)
        model = models.get_fully_connected_model(input_dim=512, seed=config["init.seed"], device='cuda', output_dim=10)
        wrapper = WandbTrainerWrapper(
            trainer_class=IntervalTrainer,
            model=model,
            train_tasks=train_tasks,
            val_tasks=test_tasks,
            test_tasks=test_tasks,
            input_dim=512
        )
        wrapper.run(project="certified-continual-learning", tags=["final_cifar_new"] + ([tag] if config["ours"] else []) + benchmark_tags, config=config, unbias_domain=[X_min, X_max])

run_cil()


Contexts: [[2, 8], [6, 1], [3, 9], [5, 0], [4, 7]]


/tmp/ipykernel_2232659/2243627393.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_2232659/2243627393.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Training Epochs: 100%|███████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.06it/s]


Test Results: [(0.4646, 0.8395), (17.0623, 0.0), (19.7875, 0.0), (16.7346, 0.0), (15.4369, 0.0)] (Avg: (13.8972, 0.1679))
Initial acc constraint violation: -0.1952 (Positive = violated)
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.66
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.85,  Min acc soft=0.85


100%|███████████████████████████████| 200/200 [00:04<00:00, 47.59it/s, size=1082.17, obj=0.210, min_soft_acc=0.657]


...

Training Epochs: 100%|██████████████████████████████| 5/5 [00:06<00:00,  1.23s/it, val_loss=2.2115, val_acc=0.2220]


Test Results: [(1.6166, 0.454), (4.0491, 0.1295), (7.1473, 0.0055), (1.8564, 0.2975), (116.942, 0.0)] (Avg: (26.3223, 0.1773))
Failed to set context: Last model layer is not a context head.


Training Epochs: 100%|██████████████████████████████| 5/5 [00:07<00:00,  1.40s/it, val_loss=3.4443, val_acc=0.1005]


Test Results: [(2.2768, 0.3245), (3.1532, 0.1815), (5.3934, 0.082), (4.9981, 0.0275), (1.9552, 0.422)] (Avg: (3.5553, 0.2075))


final_avg_accuracy,▁
final_avg_loss,▁
final_num_tasks,▁
final_total_accuracy,▁
second_task_accuracy,▁
seed,▁
final_avg_accuracy,0.1823
final_avg_loss,11.2029
final_num_tasks,5
final_total_accuracy,0.9115
second_task_accuracy,0.1615


In [11]:
def run_dil():
    animals_mask = torch.tensor([0, 0, 1, 1, 1, 1, 1, 1 ,0, 0]).to(device)

    def domain_map_fn(y):
        return animals_mask[y]
    config = {
        "ours": True,
        "init.projection_strategy": "best_loss",
        "init.n_certificate_samples": 400,
        "init.min_acc_limit": 1,
        "init.min_acc_increment": 0.2,
        "init.paradigm": "DIL",
        "init.n_iters": 200,
        "init.primal_learning_rate": 0.33,
        "init.dual_learning_rate": 0.01,
        "init.penalty_coefficient": 1,
        "init.checkpoint": 2,
        "train.l2_lambda": 0.01,
        "train.unbias_lambda": 0.01,
        "train.lr": 0.02,
        "train.weight_decay": 0,
        "train.epochs": 5,
        "train.batch_size": 128,
        "simple_train.epochs": 5,
        "simple_train.batch_size": 128,
        "simple_train.lr": 0.02,
        "simple_train.weight_decay": 0,
        "benchmarks": {
            "ewc": {
                "lmbd": 1e6,
                "fisher_batch": 64,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "sgd": {
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "lwf": {
                "lmbd": 0.1,
                "temp": 2,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "icn": {"lr": 0.01, "batch_size": 128, "epochs": 30, "lid_lr": 100}
        }
    }
    tag = "final_cifar_dil_new"
    benchmark_tags = [f"final_cifar_dil_{bench}" for bench in config["benchmarks"].keys()]

    for i in range(10):
        set_seed(i)
        config["init.seed"] = i
        train_tasks, test_tasks = get_tasks(encoder)
        model = models.get_fully_connected_model(input_dim=512, seed=config["init.seed"], device='cuda', output_dim=2)
        wrapper = WandbTrainerWrapper(
            trainer_class=IntervalTrainer,
            model=model,
            train_tasks=train_tasks,
            val_tasks=test_tasks,
            test_tasks=test_tasks,
            domain_map_fn=domain_map_fn,
            input_dim=512
        )
        wrapper.run(project="certified-continual-learning", tags=["final_cifar_new"] + ([tag] if config["ours"] else []) + benchmark_tags, config=config, unbias_domain=[X_min, X_max])

run_dil()

Contexts: [[5, 0], [4, 1], [3, 9], [7, 8], [2, 6]]


/tmp/ipykernel_2232659/2243627393.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_2232659/2243627393.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()


Training Epochs: 100%|███████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.64it/s]


Test Results: [(0.3891, 0.886), (2.2517, 0.593), (1.9727, 0.6185), (0.8951, 0.778), (1.2683, 0.7245)] (Avg: (1.3554, 0.7200))
Initial acc constraint violation: -0.1975 (Positive = violated)
Number of model parameters: 1026
Computing Rashomon set with min acc limit: 0.69
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.89,  Min acc soft=0.89


100%|█████████████████████████████████| 200/200 [00:04<00:00, 45.31it/s, size=15.53, obj=0.015, min_soft_acc=0.665]


...

Training Epochs: 100%|██████████████████████████████| 5/5 [00:05<00:00,  1.02s/it, val_loss=1.5824, val_acc=0.7420]


Test Results: [(0.8493, 0.828), (2.1911, 0.676), (1.8137, 0.737), (1.5907, 0.741), (0.9763, 0.813)] (Avg: (1.4842, 0.7590))


Training Epochs: 100%|██████████████████████████████| 5/5 [00:05<00:00,  1.13s/it, val_loss=0.9686, val_acc=0.8140]


Test Results: [(0.8495, 0.8285), (2.1914, 0.676), (1.8145, 0.737), (1.5913, 0.741), (0.975, 0.813)] (Avg: (1.4843, 0.7591))


final_avg_accuracy,▁
final_avg_loss,▁
final_num_tasks,▁
final_total_accuracy,▁
second_task_accuracy,▁
seed,▁
final_avg_accuracy,0.7762
final_avg_loss,0.80276
final_num_tasks,5
final_total_accuracy,3.881
second_task_accuracy,0.705


In [10]:
def run_til():
    config = {
        "ours": True,
        "init.projection_strategy": "best_loss",
        "init.n_certificate_samples": 400,
        "init.min_acc_limit": 1,
        "init.min_acc_increment": 0.2,
        "init.paradigm": "TIL",
        "init.n_iters": 200,
        "init.primal_learning_rate": 0.33,
        "init.dual_learning_rate": 0.01,
        "init.penalty_coefficient": 1,
        "init.checkpoint": 2,
        "train.l2_lambda": 0.01,
        "train.unbias_lambda": 0.01,
        "train.lr": 0.02,
        "train.weight_decay": 0,
        "train.epochs": 5,
        "train.batch_size": 128,
        "simple_train.epochs": 5,
        "simple_train.batch_size": 128,
        "simple_train.lr": 0.02,
        "simple_train.weight_decay": 0,
        "benchmarks": {
            "ewc": {
                "lmbd": 1e6,
                "fisher_batch": 64,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "sgd": {
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "lwf": {
                "lmbd": 0.1,
                "temp": 2,
                "lr": 0.02,
                "weight_decay": 0,
                "epochs": 5,
                "batch_size": 128,
            },
            "icn": {"lr": 0.01, "batch_size": 128, "epochs": 30, "lid_lr": 100}
        }
    }
    tag = "final_cifar_til_new"
    benchmark_tags = [f"final_cifar_til_{bench}" for bench in config["benchmarks"].keys()]

    for i in range(8, 10):
        set_seed(i)
        config["init.seed"] = i
        train_tasks, test_tasks = get_tasks(encoder)
        context_sets = get_context_sets(test_tasks)
        head = utils.InContextHead(context_sets, context_dim=10, device=device)
        model = models.get_fully_connected_model(input_dim=512, seed=config["init.seed"], device='cuda', output_dim=10, head=head)
        wrapper = WandbTrainerWrapper(
            trainer_class=IntervalTrainer,
            model=model,
            train_tasks=train_tasks,
            val_tasks=test_tasks,
            test_tasks=test_tasks,
            input_dim=512
        )
        wrapper.run(project="certified-continual-learning", tags=["final_cifar_new"] + ([tag] if config["ours"] else []) + benchmark_tags, config=config, unbias_domain=[X_min, X_max])

run_til()

Contexts: [[3, 9], [7, 8], [4, 0], [6, 1], [5, 2]]


/tmp/ipykernel_2232659/2243627393.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(train_labels, dtype=torch.int64).squeeze()
/tmp/ipykernel_2232659/2243627393.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_labels = torch.tensor(test_labels, dtype=torch.int64).squeeze()
wandb: Currently logged in as: le24 (agt-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training Epochs: 100%|███████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.73it/s]


Test Results: [(0.6837, 0.7655), (2.3045, 0.5), (2.3036, 0.0), (2.3028, 0.0), (2.3032, 0.0)] (Avg: (1.9796, 0.2531))
Initial acc constraint violation: -0.2088 (Positive = violated)
Number of model parameters: 5130
Computing Rashomon set with min acc limit: 0.58
Initial bbox:  Obj=16.00,  Size=82080.00,  Min acc hard=0.79,  Min acc soft=0.79


100%|█████████████████████████████| 200/200 [00:05<00:00, 38.33it/s, size=82085.50, obj=16.001, min_soft_acc=0.724]


...

Training Epochs: 100%|██████████████████████████████| 5/5 [00:06<00:00,  1.35s/it, val_loss=1.7151, val_acc=0.7200]


Test Results: [(1.0825, 0.664), (1.5691, 0.732), (1.7148, 0.6885), (1.4485, 0.781), (2.361, 0.413)] (Avg: (1.6352, 0.6557))


Training Epochs: 100%|██████████████████████████████| 5/5 [00:07<00:00,  1.50s/it, val_loss=2.5633, val_acc=0.4765]


Test Results: [(1.0825, 0.664), (1.5691, 0.732), (1.7148, 0.6885), (1.4485, 0.781), (1.6004, 0.6645)] (Avg: (1.4831, 0.7060))


final_avg_accuracy,▁
final_avg_loss,▁
final_num_tasks,▁
final_total_accuracy,▁
second_task_accuracy,▁
seed,▁
final_avg_accuracy,0.8246
final_avg_loss,0.49276
final_num_tasks,5
final_total_accuracy,4.123
second_task_accuracy,0.843
